In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [29]:
# import the datasets
centroid = pd.read_csv(r"data/district wise centroids.csv")
parameters = pd.read_csv(r"data/india-districts-census-2011-selected-columns.csv")

In [30]:
# basic checks
print(centroid.shape)
print(parameters.shape)

(594, 4)
(640, 28)


In [31]:
# basic checks for missing values and incorrect datatype
centroid.info()

<class 'pandas.DataFrame'>
RangeIndex: 594 entries, 0 to 593
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   State      594 non-null    str    
 1   District   594 non-null    str    
 2   Latitude   594 non-null    float64
 3   Longitude  594 non-null    float64
dtypes: float64(2), str(2)
memory usage: 29.4 KB


In [32]:
# check for duplicate values
centroid.duplicated().sum()

np.int64(0)

In [33]:
# basic checks for missing values and incorrect datatype
# check for duplicate values
parameters.info()

<class 'pandas.DataFrame'>
RangeIndex: 640 entries, 0 to 639
Data columns (total 28 columns):
 #   Column                            Non-Null Count  Dtype
---  ------                            --------------  -----
 0   District code                     640 non-null    int64
 1   State name                        640 non-null    str  
 2   District name                     640 non-null    str  
 3   Population                        640 non-null    int64
 4   Male                              640 non-null    int64
 5   Female                            640 non-null    int64
 6   Literate                          640 non-null    int64
 7   Male_Literate                     640 non-null    int64
 8   Female_Literate                   640 non-null    int64
 9   Hindus                            640 non-null    int64
 10  Muslims                           640 non-null    int64
 11  Christians                        640 non-null    int64
 12  Sikhs                             640 non-null 

In [34]:
parameters.duplicated().sum()

np.int64(0)

In [35]:
parameters.rename(columns={'District name':'District','State name':'State'},inplace=True)

In [41]:
parameters['State'] = parameters['State'].str.strip().str.lower()

In [42]:
parameters['District'] = parameters['District'].str.strip().str.lower()

In [45]:
centroid['State'] = centroid['State'].str.strip().str.lower()
centroid['District'] = centroid['District'].str.strip().str.lower()

In [46]:
centroid.head()

,State,District,Latitude,Longitude
0,andaman and nicobar,andaman islands,12.382571,92.822911
1,andaman and nicobar,nicobar islands,7.835291,93.511601
2,andhra pradesh,adilabad,19.284514,78.813212
3,andhra pradesh,anantapur,14.312066,77.460158
4,andhra pradesh,chittoor,13.331093,78.927639


In [52]:
sorted(centroid['State'].unique().tolist())

['andaman and nicobar',
 'andhra pradesh',
 'arunachal pradesh',
 'assam',
 'bihar',
 'chandigarh',
 'chhattisgarh',
 'dadra and nagar haveli',
 'daman and diu',
 'delhi',
 'goa',
 'gujarat',
 'haryana',
 'himachal pradesh',
 'jammu and kashmir',
 'jharkhand',
 'karnataka',
 'kerala',
 'lakshadweep',
 'madhya pradesh',
 'maharashtra',
 'manipur',
 'meghalaya',
 'mizoram',
 'nagaland',
 'orissa',
 'puducherry',
 'punjab',
 'rajasthan',
 'sikkim',
 'tamil nadu',
 'tripura',
 'uttar pradesh',
 'uttaranchal',
 'west bengal']

In [75]:
print(list(set(centroid['State'])-set(parameters['State'])))
print(list(set(parameters['State'])-set(centroid['State'])))

['puducherry', 'uttaranchal', 'andaman and nicobar', 'delhi']
['andaman and nicobar islands', 'nct of delhi', 'uttarakhand', 'pondicherry']


In [72]:
centroid['State'][~centroid['State'].isin(parameters['State'])].unique().tolist()

['andaman and nicobar', 'delhi', 'puducherry', 'uttaranchal']

In [76]:
state_map = {
  'andaman and nicobar':'andaman and nicobar islands',
  'delhi': 'nct of delhi',
  'pondicherry' : 'puducherry',
  'uttaranchal': 'uttarakhand'
 }

In [78]:
centroid['State'] = centroid['State'].replace(state_map)

In [79]:
parameters['State'] = parameters['State'].replace(state_map)

In [80]:
print(list(set(centroid['State'])-set(parameters['State'])))
print(list(set(parameters['State'])-set(centroid['State'])))

[]
[]


In [ ]:
# currently not doing district correction
temp_1 = sorted(list(set(centroid['District'])-set(parameters['District'])))
temp_2 = sorted(list(set(parameters['District'])-set(centroid['District'])))

In [98]:
# merge the 2 dataframes
final_df = centroid.merge(parameters,on=['District'],how='inner')

In [101]:
final_df = final_df.drop(columns=['State_y']).rename(columns={'State_x':'State'})

In [111]:
final_df.columns = [name.lower() for name in final_df.columns.tolist()]

In [113]:
# export the dataframe
final_df.to_parquet('final_df.parquet')

In [120]:
final_df.columns.tolist()[5:]

['population',
 'male',
 'female',
 'literate',
 'male_literate',
 'female_literate',
 'hindus',
 'muslims',
 'christians',
 'sikhs',
 'buddhists',
 'jains',
 'others_religions',
 'lpg_or_png_households',
 'housholds_with_electric_lighting',
 'households_with_internet',
 'households_with_computer',
 'rural_households',
 'urban_households',
 'primary_education',
 'secondary_education',
 'higher_education',
 'age_group_0_29',
 'age_group_30_49',
 'age_group_50']

In [122]:
final_df['literacy_rate'] = round((final_df['literate']/final_df['population'])*100)

In [123]:
final_df.head()

,state,district,latitude,longitude,district code,population,male,female,literate,male_literate,...,households_with_computer,rural_households,urban_households,primary_education,secondary_education,higher_education,age_group_0_29,age_group_30_49,age_group_50,literacy_rate
0,andhra pradesh,adilabad,19.284514,78.813212,532,2741239,1369597,1371642,1483347,856350,...,45202,597466,220248,344167,225957,143143,1536378,704551,422522,54.0
1,andhra pradesh,anantapur,14.312066,77.460158,553,4081148,2064495,2016653,2310960,1338474,...,48299,902997,341431,720357,331793,162175,2189396,1104565,739612,57.0
2,andhra pradesh,chittoor,13.331093,78.927639,554,4174064,2090204,2083860,2667878,1484794,...,67087,907970,395900,770836,548336,265199,2148179,1159053,807430,64.0
3,andhra pradesh,east godavari,16.782718,82.243207,545,5154296,2569688,2584608,3288577,1716933,...,93784,1199479,406362,947606,521562,271809,2724328,1478954,948122,64.0
4,andhra pradesh,guntur,15.884926,80.586576,548,4887813,2440521,2447292,2960441,1634726,...,80434,1069877,513113,845810,453446,246580,2583462,1371811,928462,61.0


In [124]:
final_df['sex_ratio'] = round((final_df['female']/final_df['male'])*1000)

In [125]:
final_df.head()

,state,district,latitude,longitude,district code,population,male,female,literate,male_literate,...,rural_households,urban_households,primary_education,secondary_education,higher_education,age_group_0_29,age_group_30_49,age_group_50,literacy_rate,sex_ratio
0,andhra pradesh,adilabad,19.284514,78.813212,532,2741239,1369597,1371642,1483347,856350,...,597466,220248,344167,225957,143143,1536378,704551,422522,54.0,1001.0
1,andhra pradesh,anantapur,14.312066,77.460158,553,4081148,2064495,2016653,2310960,1338474,...,902997,341431,720357,331793,162175,2189396,1104565,739612,57.0,977.0
2,andhra pradesh,chittoor,13.331093,78.927639,554,4174064,2090204,2083860,2667878,1484794,...,907970,395900,770836,548336,265199,2148179,1159053,807430,64.0,997.0
3,andhra pradesh,east godavari,16.782718,82.243207,545,5154296,2569688,2584608,3288577,1716933,...,1199479,406362,947606,521562,271809,2724328,1478954,948122,64.0,1006.0
4,andhra pradesh,guntur,15.884926,80.586576,548,4887813,2440521,2447292,2960441,1634726,...,1069877,513113,845810,453446,246580,2583462,1371811,928462,61.0,1003.0


In [128]:
state = 'andhra pradesh'
final_df.query('state == @state')

,state,district,latitude,longitude,district code,population,male,female,literate,male_literate,...,rural_households,urban_households,primary_education,secondary_education,higher_education,age_group_0_29,age_group_30_49,age_group_50,literacy_rate,sex_ratio
0,andhra pradesh,adilabad,19.284514,78.813212,532,2741239,1369597,1371642,1483347,856350,...,597466,220248,344167,225957,143143,1536378,704551,422522,54.0,1001.0
1,andhra pradesh,anantapur,14.312066,77.460158,553,4081148,2064495,2016653,2310960,1338474,...,902997,341431,720357,331793,162175,2189396,1104565,739612,57.0,977.0
2,andhra pradesh,chittoor,13.331093,78.927639,554,4174064,2090204,2083860,2667878,1484794,...,907970,395900,770836,548336,265199,2148179,1159053,807430,64.0,997.0
3,andhra pradesh,east godavari,16.782718,82.243207,545,5154296,2569688,2584608,3288577,1716933,...,1199479,406362,947606,521562,271809,2724328,1478954,948122,64.0,1006.0
4,andhra pradesh,guntur,15.884926,80.586576,548,4887813,2440521,2447292,2960441,1634726,...,1069877,513113,845810,453446,246580,2583462,1371811,928462,61.0,1003.0
5,andhra pradesh,hyderabad,17.389716,78.466689,536,3943323,2018575,1924748,2892155,1542688,...,0,1066289,596591,811501,517425,2249130,1068243,542669,73.0,954.0
6,andhra pradesh,karimnagar,18.515483,79.437654,534,3776269,1880800,1895469,2206829,1257238,...,877236,288044,453505,463403,293493,1940234,1031735,728494,58.0,1008.0
7,andhra pradesh,khammam,17.683921,80.670086,541,2797370,1390988,1406382,1630234,901640,...,669262,201363,346118,238811,154774,1530602,774583,477955,58.0,1011.0
8,andhra pradesh,krishna,16.143873,81.148051,547,4517398,2267375,2250023,3009718,1598959,...,909292,567042,692968,531821,312753,2340943,1284630,833959,67.0,992.0
9,andhra pradesh,kurnool,15.437597,77.900868,552,4053463,2039227,2014236,2127161,1246369,...,783853,306982,672951,271828,152453,2368684,1053618,609027,52.0,988.0
